In [ ]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
        print('{} memory growth: {}'.format(device, tf.config.experimental.get_memory_growth(device)))
else:
    print("Not enough GPU hardware devices available")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
import json
import matplotlib.ticker as mtick

output_dir = "../data/train/RFmodels/results"
model_dir = os.path.join(output_dir, "models")

os.makedirs(output_dir, exist_ok=True)
os.makedirs("../data/train/RFmodels/measure", exist_ok = True)
#Download csv file from Zenodo.
external_data_path = "..data/train/RFmodels/measure/RF_Culture_test.csv" 

df_external = pd.read_csv(external_data_path)
X_external = df_external[feature_names]

y_external_true = df_external["CellType"]

external_results = []
external_confusion_matrices = []
sum_external_cm = np.zeros((len(rf_classifier.classes_), len(rf_classifier.classes_)), dtype=int)  

loaded_models = []

for i in range(n_splits):
    model_path = os.path.join(model_dir, f"RFmodel_fold{i+1}.pkl")
    with open(model_path, 'rb') as f:
        loaded_model = pickle.load(f)
        loaded_models.append(loaded_model)

for fold, model in enumerate(loaded_models):

    y_external_pred = model.predict(X_external)

    report = classification_report(y_external_true, y_external_pred, output_dict=True)
    accuracy = accuracy_score(y_external_true, y_external_pred)

    external_results.append({
        'fold': fold + 1,
        'classification_report': report,
        'accuracy': accuracy,
    })

    cm = confusion_matrix(y_external_true, y_external_pred, labels=rf_classifier.classes_)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_classifier.classes_)
    plt.figure(figsize=(10, 7))
    disp.plot(cmap = "Blues")
    plt.title(f"External Confusion Matrix - Fold {fold+1}")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"external_confusion_matrix_fold{fold+1}.tif"), dpi=600, transparent=True)
    plt.close()

    cm_percentage = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100  
    cm_percentage = np.round(cm_percentage, decimals=2)  
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm_percentage, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=rf_classifier.classes_, yticklabels=rf_classifier.classes_)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"External Confusion Matrix - Fold {fold+1} (Percentage)")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"external_confusion_matrix_fold{fold+1}_percentage.tif"), dpi=600, transparent=True)
    plt.close() 

    external_confusion_matrices.append(cm)  
    sum_external_cm += cm 

plt.figure(figsize=(10, 7))
sns.heatmap(sum_external_cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=rf_classifier.classes_, yticklabels=rf_classifier.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Sum of External Confusion Matrices (Counts)")
plt.savefig(os.path.join(output_dir, "sum_external_confusion_matrix_counts.tif"), dpi=600, transparent=True)

sum_external_cm_percentage = sum_external_cm.astype('float') / sum_external_cm.sum(axis=1)[:, np.newaxis] * 100
sum_external_cm_percentage = np.round(sum_external_cm_percentage, decimals = 2)

plt.figure(figsize=(10, 7))
sns.heatmap(sum_external_cm_percentage, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=rf_classifier.classes_, yticklabels=rf_classifier.classes_)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Sum of External Confusion Matrices (Percentage)")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "sum_external_confusion_matrix_percentage.tif"), dpi=600, transparent=True)
plt.close('all')

with open(os.path.join(output_dir,"external_results.json"), 'w') as f: 
    json.dump(external_results, f, indent=4, default=str)